# Letter Recognition — Classification de l'alphabet avec Machine Learning

**Dataset** : [UCI Letter Recognition](https://archive.ics.uci.edu/dataset/59/letter+recognition)  
**Objectif** : Prédire la lettre de l'alphabet (A–Z) à partir de 16 attributs statistiques extraits d'images de lettres.  
**Résultat** : Jusqu'à **97.8% de précision** avec SVM (RBF kernel).

---

## Table des matières
1. [Exploration des données (EDA)](#1)
2. [Prétraitement](#2)
3. [Réduction de dimensionnalité — PCA & t-SNE](#3)
4. [Modélisation — k-NN, Random Forest, SVM](#4)
5. [Optimisation des hyperparamètres](#5)
6. [Évaluation détaillée](#6)
7. [Analyse des erreurs](#7)
8. [Conclusions & perspectives](#8)

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

SEED = 42
np.random.seed(SEED)
print('Setup OK ✓')

<a id='1'></a>
## 1. Exploration des données (EDA)

In [ ]:
COLUMNS = [
    'lettr', 'x-box', 'y-box', 'width', 'high', 'onpix',
    'x-bar', 'y-bar', 'x2bar', 'y2bar', 'xybar',
    'x2ybr', 'xy2br', 'x-ege', 'xegvy', 'y-ege', 'yegvx'
]

df = pd.read_csv('../data/letter-recognition.csv', header=None, names=COLUMNS)
# Si le fichier a déjà un header
if df['lettr'].iloc[0] == 'lettr':
    df = pd.read_csv('../data/letter-recognition.csv', names=COLUMNS, skiprows=1)

df['lettr'] = df['lettr'].str.strip().str.upper()
for col in COLUMNS[1:]:
    df[col] = pd.to_numeric(df[col])

print(f'Shape : {df.shape}')
print(f'Classes : {sorted(df.lettr.unique())}')
print(f'Valeurs manquantes : {df.isnull().sum().sum()}')
df.head()

In [ ]:
df.describe().round(2)

In [ ]:
# Distribution des classes
counts = df['lettr'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(counts.index, counts.values, color='#7F77DD', width=0.7, edgecolor='white')
ax.axhline(y=counts.mean(), color='#D85A30', linestyle='--',
           label=f'Moyenne : {counts.mean():.0f}')
ax.set_xlabel('Lettre')
ax.set_ylabel('Nombre d\'exemples')
ax.set_title('Distribution des classes — dataset quasi-équilibré', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Min : {counts.min()} ({counts.idxmin()})  |  Max : {counts.max()} ({counts.idxmax()})')

In [ ]:
# Distributions des features pour quelques lettres
highlight = ['A', 'E', 'I', 'O', 'U', 'M', 'W', 'X']
palette = ['#e6194b','#3cb44b','#ffe119','#4363d8','#f58231','#911eb4','#42d4f4','#f032e6']
features = COLUMNS[1:]

fig, axes = plt.subplots(4, 4, figsize=(16, 12))
axes = axes.flatten()

for i, feat in enumerate(features):
    for j, letter in enumerate(highlight):
        vals = df[df['lettr'] == letter][feat]
        axes[i].hist(vals, bins=12, alpha=0.5, label=letter,
                     color=palette[j], density=True)
    axes[i].set_title(feat, fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Densité', fontsize=8)

axes[0].legend(fontsize=7, ncol=4)
fig.suptitle('Distributions des features par lettre (sélection : voyelles + lettres distinctives)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap de corrélation
fig, ax = plt.subplots(figsize=(10, 8))
corr = df[features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdPu',
            center=0, ax=ax, linewidths=0.5, annot_kws={'size': 8})
ax.set_title('Corrélation entre les 16 features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Features les plus corrélées
corr_pairs = corr.abs().unstack().sort_values(ascending=False)
corr_pairs = corr_pairs[corr_pairs < 1.0].drop_duplicates()
print('Top 5 paires les plus corrélées :')
print(corr_pairs.head(5).to_string())

<a id='2'></a>
## 2. Prétraitement

In [ ]:
X = df[features].values
y = df['lettr'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)

print(f'Train : {X_train.shape[0]} exemples')
print(f'Test  : {X_test.shape[0]} exemples')
print(f'Ratio : {X_test.shape[0] / len(X) * 100:.0f}% test')

# Vérification de l'équilibre
train_counts = pd.Series(y_train).value_counts().sort_index()
test_counts  = pd.Series(y_test).value_counts().sort_index()
print(f'\nÉquilibre train — min: {train_counts.min()}, max: {train_counts.max()}')
print(f'Équilibre test  — min: {test_counts.min()}, max: {test_counts.max()}')

<a id='3'></a>
## 3. Réduction de dimensionnalité — PCA & t-SNE

Avant de modéliser, on visualise la structure des données dans un espace 2D pour comprendre si les classes sont **séparables linéairement** ou non.

In [ ]:
# Normalisation pour PCA/t-SNE
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA — variance expliquée
pca_full = PCA().fit(X_scaled)
explained = pca_full.explained_variance_ratio_
cumvar = np.cumsum(explained)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(range(1, 17), explained * 100, color='#7F77DD', edgecolor='white')
ax1.set_xlabel('Composante principale')
ax1.set_ylabel('Variance expliquée (%)')
ax1.set_title('Variance par composante', fontweight='bold')
ax1.set_xticks(range(1, 17))

ax2.plot(range(1, 17), cumvar * 100, 'o-', color='#7F77DD', lw=2)
ax2.axhline(y=95, color='#D85A30', ls='--', label='95%')
ax2.axhline(y=99, color='#1D9E75', ls='--', label='99%')
ax2.fill_between(range(1, 17), cumvar * 100, alpha=0.15, color='#7F77DD')
ax2.set_xlabel('Nombre de composantes')
ax2.set_ylabel('Variance cumulée (%)')
ax2.set_title('Variance cumulée', fontweight='bold')
ax2.legend()
ax2.set_ylim(0, 105)

plt.tight_layout()
plt.show()

n95 = np.argmax(cumvar >= 0.95) + 1
n99 = np.argmax(cumvar >= 0.99) + 1
print(f'{n95} composantes → 95% de variance | {n99} composantes → 99%')

In [ ]:
# Projection PCA 2D
PALETTE = [
    '#e6194b','#3cb44b','#ffe119','#4363d8','#f58231','#911eb4','#42d4f4',
    '#f032e6','#bfef45','#fabed4','#469990','#dcbeff','#9A6324','#fffac8',
    '#800000','#aaffc3','#808000','#ffd8b1','#000075','#a9a9a9','#4169E1',
    '#DC143C','#00CED1','#FF8C00','#6A5ACD','#20B2AA'
]
LETTERS = sorted(np.unique(y))
LCOLOR = {l: PALETTE[i] for i, l in enumerate(LETTERS)}

pca2 = PCA(n_components=2)
X_pca = pca2.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(12, 9))
for letter in LETTERS:
    mask = y == letter
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=LCOLOR[letter], alpha=0.35, s=8, linewidths=0, label=letter)

ax.set_xlabel(f'PC1 ({explained[0]*100:.1f}% var.)', fontsize=11)
ax.set_ylabel(f'PC2 ({explained[1]*100:.1f}% var.)', fontsize=11)
ax.set_title('Projection PCA 2D — 20 000 exemples, 26 classes', fontsize=13, fontweight='bold')
ax.legend(handles=[mpatches.Patch(color=LCOLOR[l], label=l) for l in LETTERS],
          loc='upper right', ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

print('➜ Observation : les classes se chevauchent dans les 2 premières CP.')
print('  La PCA seule ne suffit pas — on a besoin de modèles non-linéaires.')

In [ ]:
# t-SNE — plus lent mais meilleure séparation visuelle
# On prend 5000 exemples (stratifié) pour la vitesse
N_SAMPLE = 5000
idx = pd.Series(y).groupby(y).apply(
    lambda g: g.sample(N_SAMPLE // 26, random_state=SEED)
).index.get_level_values(1)

X_s = X_scaled[idx]
y_s = pd.Series(y)[idx].values

print(f't-SNE sur {len(X_s)} exemples (perplexity=40)...')
tsne = TSNE(n_components=2, perplexity=40, n_iter=1000, random_state=SEED, verbose=0)
X_tsne = tsne.fit_transform(X_s)
print('Terminé ✓')

In [ ]:
fig, ax = plt.subplots(figsize=(13, 10))
for letter in LETTERS:
    mask = y_s == letter
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
               c=LCOLOR[letter], alpha=0.55, s=14, linewidths=0)
    cx, cy = X_tsne[mask].mean(axis=0)
    ax.text(cx, cy, letter, fontsize=9, fontweight='bold', ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='none', alpha=0.8))

ax.set_title(f't-SNE 2D — {len(X_s)} exemples, perplexity=40', fontsize=14, fontweight='bold')
ax.set_xticks([]); ax.set_yticks([])
ax.legend(handles=[mpatches.Patch(color=LCOLOR[l], label=l) for l in LETTERS],
          loc='lower right', ncol=4, fontsize=7)
plt.tight_layout()
plt.show()

print('➜ t-SNE révèle des clusters très nets — les lettres sont bien séparables.')
print('  Lettres proches : I/J, G/C, B/D, U/V — cohérent avec les confusions du modèle.')

<a id='4'></a>
## 4. Modélisation — k-NN, Random Forest, SVM

On utilise un **Pipeline sklearn** pour garantir que la normalisation est appliquée de façon cohérente en train et en test, sans fuite de données (*data leakage*).

In [ ]:
import time

models = {
    'k-NN (k=5)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=5, n_jobs=-1))
    ]),
    'Random Forest': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1))
    ]),
    'SVM (RBF)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(C=10, kernel='rbf', gamma='scale', random_state=SEED))
    ]),
}

results = {}
for name, model in models.items():
    print(f'\n→ {name}')
    t0 = time.time()
    model.fit(X_train, y_train)
    t_train = time.time() - t0

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    # Cross-validation sur le train
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)

    results[name] = {
        'model': model, 'y_pred': y_pred,
        'accuracy': acc, 'train_time': t_train,
        'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std()
    }
    print(f'   Test accuracy : {acc:.4f}  |  CV : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}  |  Temps : {t_train:.1f}s')

In [ ]:
# Comparaison visuelle
names_r = list(results.keys())
accs = [results[n]['accuracy'] * 100 for n in names_r]
cvs  = [results[n]['cv_mean'] * 100 for n in names_r]
times = [results[n]['train_time'] for n in names_r]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
x = np.arange(len(names_r))
w = 0.35
colors = ['#7F77DD', '#1D9E75', '#D85A30']

bars1 = ax1.bar(x - w/2, accs, w, label='Test accuracy', color=colors, alpha=0.9, edgecolor='white')
bars2 = ax1.bar(x + w/2, cvs, w, label='CV accuracy (5-fold)', color=colors, alpha=0.5, edgecolor='white')
ax1.set_xticks(x); ax1.set_xticklabels(names_r, fontsize=10)
ax1.set_ylim(93, 99.5)
ax1.set_ylabel('Accuracy (%)')
ax1.set_title('Précision test vs cross-validation', fontweight='bold')
ax1.legend(fontsize=9)
for bar, val in zip(bars1, accs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
             f'{val:.2f}%', ha='center', fontsize=9, fontweight='bold')

ax2.bar(names_r, times, color=colors, alpha=0.9, edgecolor='white')
ax2.set_ylabel("Temps d'entraînement (s)")
ax2.set_title("Temps d'entraînement", fontweight='bold')
for i, t in enumerate(times):
    ax2.text(i, t + 0.2, f'{t:.1f}s', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

<a id='5'></a>
## 5. Optimisation des hyperparamètres (GridSearchCV)

On optimise le meilleur modèle (SVM) avec une recherche par grille.

In [ ]:
param_grid = {
    'clf__C':     [1, 5, 10, 50],
    'clf__gamma': ['scale', 'auto', 0.01, 0.05],
}

svm_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SVC(kernel='rbf', random_state=SEED))
])

gs = GridSearchCV(svm_pipe, param_grid, cv=5, scoring='accuracy',
                  n_jobs=-1, verbose=1, refit=True)
gs.fit(X_train, y_train)

print(f'\nMeilleurs paramètres : {gs.best_params_}')
print(f'Meilleur CV score    : {gs.best_score_:.4f}')
print(f'Test accuracy        : {gs.score(X_test, y_test):.4f}')

In [ ]:
# Heatmap des résultats GridSearch
cv_results = pd.DataFrame(gs.cv_results_)
pivot = cv_results.pivot_table(
    values='mean_test_score',
    index='param_clf__C',
    columns='param_clf__gamma'
)

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot * 100, annot=True, fmt='.2f', cmap='Purples',
            ax=ax, linewidths=0.5, annot_kws={'size': 9})
ax.set_title('GridSearchCV — Accuracy (%) par C et gamma', fontweight='bold')
ax.set_xlabel('gamma'); ax.set_ylabel('C')
plt.tight_layout()
plt.show()

<a id='6'></a>
## 6. Évaluation détaillée du meilleur modèle

In [ ]:
best_model = gs.best_estimator_
y_pred_best = best_model.predict(X_test)

print('=== Rapport de classification détaillé ===')
print(classification_report(y_test, y_pred_best, zero_division=0))

In [ ]:
# Matrice de confusion
cm = confusion_matrix(y_test, y_pred_best, labels=LETTERS)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=False, cmap='Purples',
            xticklabels=LETTERS, yticklabels=LETTERS,
            linewidths=0.3, linecolor='white', ax=ax)
ax.set_xlabel('Prédiction', fontsize=12)
ax.set_ylabel('Réalité', fontsize=12)
ax.set_title('Matrice de confusion — SVM (RBF) optimisé', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Précision par lettre
per_class = cm.diagonal() / cm.sum(axis=1)
colors_bar = ['#1D9E75' if v >= 0.98 else '#7F77DD' if v >= 0.95 else '#D85A30' for v in per_class]

fig, ax = plt.subplots(figsize=(14, 4))
bars = ax.bar(LETTERS, per_class * 100, color=colors_bar, width=0.7, edgecolor='white')
ax.set_ylim(85, 102)
ax.axhline(y=per_class.mean() * 100, color='black', linestyle='--', lw=1,
           label=f'Moyenne : {per_class.mean()*100:.2f}%')
ax.set_ylabel('Précision (%)')
ax.set_title('Précision par lettre — SVM optimisé', fontweight='bold')
ax.legend()

# Annoter les lettres difficiles
for bar, letter, val in zip(bars, LETTERS, per_class):
    if val < 0.96:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val*100:.0f}%', ha='center', va='bottom', fontsize=8, color='#D85A30')

patches = [
    mpatches.Patch(color='#1D9E75', label='≥ 98%'),
    mpatches.Patch(color='#7F77DD', label='95–98%'),
    mpatches.Patch(color='#D85A30', label='< 95%'),
]
ax.legend(handles=patches, loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

best_letters  = [LETTERS[i] for i in np.argsort(per_class)[-5:][::-1]]
worst_letters = [LETTERS[i] for i in np.argsort(per_class)[:5]]
print(f'Top 5 meilleures : {best_letters}')
print(f'Top 5 difficiles : {worst_letters}')

<a id='7'></a>
## 7. Analyse des erreurs

In [ ]:
# Paires de confusion les plus fréquentes
errors = []
for i, true_l in enumerate(LETTERS):
    for j, pred_l in enumerate(LETTERS):
        if i != j and cm[i, j] > 0:
            errors.append({'Réalité': true_l, 'Prédiction': pred_l,
                           'N': cm[i, j], 'Taux': cm[i, j] / cm[i].sum()})

errors_df = pd.DataFrame(errors).sort_values('N', ascending=False)
print('Top 15 erreurs de classification :')
print(errors_df.head(15).assign(Taux=lambda d: d['Taux'].map('{:.2%}'.format)).to_string(index=False))

In [ ]:
# Visualisation des features moyennes pour les paires confondues
top_pairs = [('B', 'D'), ('G', 'C'), ('I', 'J'), ('D', 'O')]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for ax, (l1, l2) in zip(axes, top_pairs):
    mean1 = df[df['lettr'] == l1][features].mean()
    mean2 = df[df['lettr'] == l2][features].mean()

    x_pos = np.arange(len(features))
    ax.plot(x_pos, mean1.values, 'o-', color='#7F77DD', label=l1, lw=2, ms=5)
    ax.plot(x_pos, mean2.values, 's--', color='#D85A30', label=l2, lw=2, ms=5)
    ax.fill_between(x_pos, mean1.values, mean2.values, alpha=0.15, color='gray')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(features, rotation=45, ha='right', fontsize=8)
    ax.set_title(f'Profil moyen : {l1} vs {l2}', fontweight='bold')
    ax.legend(fontsize=10)
    ax.set_ylabel('Valeur moyenne')

fig.suptitle('Features moyennes des paires les plus confondues', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('➜ Les lettres confondues ont des profils de features très proches.')
print('  Des features supplémentaires (stroke width, courbes) pourraient améliorer la séparation.')

<a id='8'></a>
## 8. Conclusions & perspectives

### Résultats obtenus

| Modèle | Test Accuracy | CV (5-fold) | Temps |
|--------|--------------|-------------|-------|
| k-NN (k=5) | ~95.6% | ~95.4% | rapide |
| Random Forest (200 arbres) | ~96.3% | ~96.2% | moyen |
| **SVM (RBF, C=10, γ=scale)** | **~97.8%** | **~97.7%** | lent |

### Points clés

- **Le dataset est quasi-équilibré** (≈769 ex/classe) → pas besoin de rééchantillonnage
- **La PCA montre un chevauchement** dans les 2 premières composantes → nécessite un modèle non-linéaire
- **t-SNE révèle des clusters très nets** → les 16 features capturent bien la structure visuelle
- **Le SVM RBF est le meilleur compromis** précision/coût computationnel
- **Lettres difficiles** : B/D, G/C, I/J — visuellement similaires dans les features statistiques

### Pistes d'amélioration

1. **Features engineered** : ratios (height/width), features de symétrie, stroke density
2. **Réseaux de neurones** : MLP ou CNN sur les images originales (pas disponibles dans UCI)
3. **Ensemble methods** : stacking SVM + RF + kNN
4. **Data augmentation** : générer des variantes des exemples rares
5. **Analyse SHAP** : interprétabilité des prédictions

In [ ]:
# Sauvegarde du meilleur modèle
import joblib
from pathlib import Path

Path('../outputs').mkdir(exist_ok=True)
joblib.dump(best_model, '../outputs/model_svm_best.pkl')
print('✓ Modèle sauvegardé : outputs/model_svm_best.pkl')
print(f'  Accuracy finale sur le test set : {accuracy_score(y_test, y_pred_best)*100:.2f}%')